In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, upper
spark = SparkSession.builder.appName("FinalProject").getOrCreate()

In [0]:
orders = spark.table("workspace.default.orders")
customers = spark.table("workspace.default.customers")
items = spark.table("workspace.default.order_items")
orders.createOrReplaceTempView("orders_bronze")
customers.createOrReplaceTempView("customers_bronze")
items.createOrReplaceTempView("items_bronze")

In [0]:
customers_silver = spark.sql("""
SELECT DISTINCT *
FROM customers_bronze
WHERE customer_name IS NOT NULL
  AND signup_date IS NOT NULL
""")

In [0]:
orders_silver = spark.sql("""
SELECT *
FROM orders_bronze
WHERE customer_id IS NOT NULL
  AND city IS NOT NULL
  AND order_date IS NOT NULL
""")

In [0]:
items_silver = spark.sql("""
SELECT DISTINCT *,
       price * quantity AS revenue
FROM items_bronze
WHERE price IS NOT NULL
  AND quantity IS NOT NULL
""")

In [0]:
customers_silver.createOrReplaceTempView("customers_silver")
orders_silver.createOrReplaceTempView("orders_silver")
items_silver.createOrReplaceTempView("items_silver")

In [0]:
orders_silver = orders_silver.withColumn(
    "city",
    upper(trim(col("city")))
)

items_silver = items_silver.withColumn(
    "price",
    when(col("price") < 0, 0).otherwise(col("price"))
)

In [0]:
sales_gold = spark.sql("""
SELECT o.order_id,
       c.customer_name,
       o.city,
       i.product,
       i.category,
       i.revenue
FROM orders_silver o
JOIN customers_silver c
  ON o.customer_id = c.customer_id
JOIN items_silver i
  ON o.order_id = i.order_id
""")

sales_gold.createOrReplaceTempView("sales_gold")

In [0]:
city_revenue = spark.sql("""
SELECT city,
       SUM(revenue) AS total_revenue
FROM sales_gold
GROUP BY city
""")

In [0]:
top_customers = spark.sql("""
SELECT *
FROM (
    SELECT customer_name,
           SUM(revenue) AS total_revenue,
           RANK() OVER (ORDER BY SUM(revenue) DESC) AS rnk
    FROM sales_gold
    GROUP BY customer_name
) t
WHERE rnk <= 3
""")

In [0]:
top_products = spark.sql("""
SELECT *
FROM (
    SELECT city,
           product,
           SUM(revenue) AS total_revenue,
           RANK() OVER (PARTITION BY city ORDER BY SUM(revenue) DESC) AS rnk
    FROM sales_gold
    GROUP BY city, product
) t
WHERE rnk = 1
""")

In [0]:
category_perf = spark.sql("""
SELECT category,
       SUM(revenue) AS total_revenue
FROM sales_gold
GROUP BY category
""").show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       144000|
|Accessories|        17800|
|    Fashion|        21800|
+-----------+-------------+



In [0]:
repeat_customers = spark.sql("""
SELECT customer_name,
       COUNT(*) AS total_orders
FROM sales_gold
GROUP BY customer_name
HAVING COUNT(*) > 1
""").show()

+-------------+------------+
|customer_name|total_orders|
+-------------+------------+
|      Koushik|           2|
|         Ravi|           2|
+-------------+------------+



In [0]:
customer_type = spark.sql("""
SELECT customer_name,
       SUM(revenue) AS total_revenue,
       CASE 
         WHEN SUM(revenue) > 50000 THEN 'HIGH VALUE'
         ELSE 'LOW VALUE'
       END AS customer_segment
FROM sales_gold
GROUP BY customer_name
""").show()

+-------------+-------------+----------------+
|customer_name|total_revenue|customer_segment|
+-------------+-------------+----------------+
|        Arjun|        22000|       LOW VALUE|
|        Sneha|         7800|       LOW VALUE|
|      Koushik|        24500|       LOW VALUE|
|         Ravi|         9200|       LOW VALUE|
|        Kiran|         3600|       LOW VALUE|
|        Pooja|         4500|       LOW VALUE|
|         Anil|        50000|       LOW VALUE|
|        Rahul|        52000|      HIGH VALUE|
|          Raj|         5000|       LOW VALUE|
|        Meena|         5000|       LOW VALUE|
+-------------+-------------+----------------+



In [0]:
city_revenue.write.mode("overwrite").saveAsTable("workspace.default.city_revenue")
top_customers.write.mode("overwrite").saveAsTable("workspace.default.top_customers")
top_products.write.mode("overwrite").saveAsTable("workspace.default.top_products")